#### 04 - Silver Layer: Dallas Food Inspections
**Medallion Architecture — Silver Zone**

Steps:
1. Install and import DQX
2. Read from Bronze
3. DQX Profile — document data quality
4. DQX Expectations — define and apply rules, split good/bad
5. Additional business rules (complex rules not in DQX)
6. Standardize schema
7. Unpivot violations wide to long
8. Write to Silver
9. Verify

- Source: `bronze.bronze_dallas_inspections`
- Target: `silver.silver_dallas_inspections` + `silver.silver_dallas_violations`

##### Step 1: Install DQX

In [0]:
%pip install databricks-labs-dqx -q

In [0]:
dbutils.library.restartPython()

##### Step 2: Configuration & Setup

In [0]:
from databricks.labs.dqx.engine import DQEngine
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, upper, initcap, to_date,
    when, lit, regexp_extract, concat_ws,
    monotonically_increasing_id, current_timestamp
)

BRONZE_TABLE = "bronze.bronze_dallas_inspections"
SILVER_DB    = "silver"
SILVER_TABLE = "silver_dallas_inspections"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {SILVER_DB}")
print(f"Source : {BRONZE_TABLE}")
print(f"Target : {SILVER_DB}.{SILVER_TABLE}")

##### Step 3: Read from Bronze

In [0]:
df_bronze = spark.table(BRONZE_TABLE)
bronze_count = df_bronze.count()
print(f"Bronze row count : {bronze_count:,}")
print(f"Columns          : {len(df_bronze.columns)}")
display(df_bronze.select(
    "restaurant_name", "inspection_type", "inspection_date",
    "inspection_score", "zip_code"
).limit(3))

##### Step 4: DQX Profiling
Profile key columns to document quality before cleansing.

In [0]:
# DQX Profiling — null count documentation
profile_cols = ["restaurant_name", "inspection_type", "inspection_date",
                "inspection_score", "zip_code"]

print("\n=== Null Count Profile ===")
for c in profile_cols:
    null_ct = df_bronze.filter(col(c).isNull()).count()
    pct = null_ct / bronze_count * 100
    print(f"  {c:25s} : {null_ct:,} nulls ({pct:.2f}%)")

print("\n=== Unique Value Counts ===")
for c in ["inspection_type"]:
    uniq = df_bronze.select(c).distinct().count()
    print(f"  {c:25s} : {uniq} unique values")

print("\n=== Inspection Score Range ===")
from pyspark.sql.functions import min, max, avg
df_bronze.select(
    min("inspection_score").alias("min_score"),
    max("inspection_score").alias("max_score"),
    avg("inspection_score").alias("avg_score")
).show()

display(df_bronze.select(profile_cols).limit(3))

##### Step 5: Pre-compute Violation Count per Row
Required for the business rule: score >= 90 must have <= 3 violations.

In [0]:
# Count non-empty violation descriptions per row
viol_count_expr = sum([
    when(
        col(f"violation_description_{i}").isNotNull() &
        (col(f"violation_description_{i}") != ""), 1
    ).otherwise(0)
    for i in range(1, 26)
])

df_with_count = df_bronze.withColumn("violation_count", viol_count_expr)

# Also build combined violation text for Urgent/Critical check
df_with_count = df_with_count.withColumn(
    "_all_violation_text",
    upper(concat_ws(" ", *[col(f"violation_description_{i}") for i in range(1, 26)]))
)

print("Violation count added. Sample:")
display(df_with_count.select(
    "restaurant_name", "inspection_score", "violation_count"
).limit(5))

##### Step 6: DQX Expectations — Define Rules
Dallas-specific rules per assignment:
- `restaurant_name` cannot be null
- `inspection_date` cannot be null
- `inspection_type` cannot be null
- `zip_code` cannot be null and must match 5-digit format
- `inspection_score` must be between 0 and 100

In [0]:
checks = [
    # Rule 1: Restaurant name not null
    {
        "name": "restaurant_name_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "restaurant_name"}
        }
    },
    # Rule 2: Inspection date not null
    {
        "name": "inspection_date_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "inspection_date"}
        }
    },
    # Rule 3: Inspection type not null
    {
        "name": "inspection_type_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "inspection_type"}
        }
    },
    # Rule 4: Zip not null
    {
        "name": "zip_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "zip_code"}
        }
    },
    # Rule 5: Score not null
    {
        "name": "score_not_null",
        "criticality": "error",
        "check": {
            "function": "is_not_null",
            "arguments": {"column": "inspection_score"}
        }
    },
]

print(f"Total DQX rules defined: {len(checks)}")
for c in checks:
    print(f"  - {c['name']}")

##### Step 7: Apply DQX Expectations — Split Good/Bad Rows

In [0]:
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
dqe = DQEngine(ws)

df_good, df_bad = dqe.apply_checks_by_metadata_and_split(
    df_with_count, checks
)

good_count = df_good.count()
bad_count  = df_bad.count()

print("=== DQX Results ===")
print(f"  Bronze rows  : {bronze_count:,}")
print(f"  ✅ Good rows : {good_count:,}")
print(f"  ❌ Bad rows  : {bad_count:,}  ({bad_count/bronze_count*100:.2f}% dropped)")

print("\nSample bad rows:")
display(df_bad.select(
    "restaurant_name", "inspection_date", "zip_code",
    "inspection_score", "_errors"
).limit(10))

##### Step 8: Apply Complex Business Rules
These cannot be expressed in standard DQX functions — applied as manual filters.
Documenting rows dropped with reason for audit trail.

In [0]:
# Rule 7: Score >= 90 must have <= 3 violations
rule7_dropped = df_good.filter(
    (col("inspection_score") >= 90) & (col("violation_count") > 3)
).count()

df_good = df_good.filter(
    ~((col("inspection_score") >= 90) & (col("violation_count") > 3))
)
print(f"Rule 7 - Score>=90 but >3 violations dropped : {rule7_dropped:,} rows")

# Rule 8: Score >= 90 (PASS) but has Urgent/Critical in violation text
rule8_dropped = df_good.filter(
    (col("inspection_score") >= 90) &
    (
        col("_all_violation_text").contains("URGENT") |
        col("_all_violation_text").contains("CRITICAL")
    )
).count()

df_good = df_good.filter(
    ~(
        (col("inspection_score") >= 90) &
        (
            col("_all_violation_text").contains("URGENT") |
            col("_all_violation_text").contains("CRITICAL")
        )
    )
)
print(f"Rule 8 - PASS but Urgent/Critical violation dropped : {rule8_dropped:,} rows")
print(f"\nFinal good rows after all rules: {df_good.count():,}")

##### Step 9: Standardize Schema

In [0]:
df_silver = (
    df_good
    .withColumn("restaurant_name",  trim(upper(col("restaurant_name"))))
    .withColumn("inspection_type",  initcap(trim(col("inspection_type"))))
    .withColumn("street_address",   trim(upper(col("street_address"))))
    .withColumn("inspection_date",  to_date(col("inspection_date"), "MM/dd/yyyy"))
    .withColumn("inspection_score", col("inspection_score").cast("int"))
    # Extract lat/long from string '(32.869, -96.770)'
    .withColumn("latitude",
        regexp_extract(col("lat_long_location"), r'\(([\d.\-]+),', 1).cast("double")
    )
    .withColumn("longitude",
        regexp_extract(col("lat_long_location"), r',\s*([\d.\-]+)\)', 1).cast("double")
    )
    .withColumn("_silver_timestamp", current_timestamp())
    .withColumn("_source_city",      lit("Dallas"))
    .drop("_errors", "_warnings", "_all_violation_text")
)

print(f"Silver row count: {df_silver.count():,}")
display(df_silver.select(
    "restaurant_name", "inspection_date", "inspection_type",
    "inspection_score", "violation_count", "zip_code",
    "latitude", "longitude"
).limit(5))

##### Step 10: Unpivot Violations — Wide to Long
Dallas has 25 violation column sets. Explode into one row per violation.

In [0]:
df_silver = df_silver.withColumn("_row_id", monotonically_increasing_id())
df_silver.createOrReplaceTempView("silver_dallas_temp")

union_parts = []
for i in range(1, 26):
    union_parts.append(f"""
        SELECT _row_id, restaurant_name, inspection_date, inspection_score,
               {i}                           AS violation_sequence,
               violation_description_{i}     AS violation_description,
               violation_points_{i}          AS violation_points,
               violation_memo_{i}            AS violation_memo,
               'Dallas'                      AS source_city
        FROM silver_dallas_temp
        WHERE violation_description_{i} IS NOT NULL
          AND violation_description_{i} != ''
    """)

df_violations = (
    spark.sql(" UNION ALL ".join(union_parts))
    .dropDuplicates(["_row_id", "violation_description"])
)

print(f"Total violation rows after unpivot: {df_violations.count():,}")
display(df_violations.limit(5))

##### Step 11: Write Silver Tables

In [0]:
# Drop wide violation columns from main table before writing
viol_cols_to_drop = [
    c for c in df_silver.columns
    if c.startswith('violation_description_') or
       c.startswith('violation_points_') or
       c.startswith('violation_memo_') or
       c.startswith('violation_detail_')
]
df_silver_clean = df_silver.drop(*viol_cols_to_drop)

# Write main inspections table
(
    df_silver_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.{SILVER_TABLE}")
)
print(f"✅ Written: {SILVER_DB}.{SILVER_TABLE}")

# Write violations long table
(
    df_violations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.silver_dallas_violations")
)
print(f"✅ Written: {SILVER_DB}.silver_dallas_violations")

##### Step 12: Verify

In [0]:
insp_count = spark.table(f"{SILVER_DB}.{SILVER_TABLE}").count()
viol_count = spark.table(f"{SILVER_DB}.silver_dallas_violations").count()

print(f"✅ Silver inspections   : {insp_count:,}")
print(f"✅ Silver violations    : {viol_count:,}")
print(f"   Bronze rows          : {bronze_count:,}")
print(f"   Total rows dropped   : {bronze_count - insp_count:,}")
print(f"   Drop rate            : {(bronze_count - insp_count)/bronze_count*100:.1f}%")

# Verify no scores > 100
df_s = spark.table(f"{SILVER_DB}.{SILVER_TABLE}")
bad_scores = df_s.filter(col("inspection_score") > 100).count()
print(f"   Scores > 100 (must be 0): {bad_scores}")

# Verify no nulls in key fields
for field in ["restaurant_name", "inspection_date", "inspection_type", "zip_code"]:
    nc = df_s.filter(col(field).isNull()).count()
    status = "✅" if nc == 0 else "❌"
    print(f"   {status} Nulls in {field}: {nc}")

display(df_s.limit(5))